# Çapraz seed analizi, 5. yöntem, kilit ve bağımsız test
1. 4 seed'in geliştirme/doğrulama referansları (sekans 01)
2. **Çapraz seed matrisi:** 4 üye yöntemi × 4 seed doğrulaması (sızıntılı kareler hariç)
3. **5. yöntem (dayanıklı pipeline):** aday havuzu, ortak geliştirme havuzunda ayar, 4 seed doğrulamasında seçim
4. 🔒 **Kilit:** baseline + 4 üye yöntemi + 5. yöntem → 60 test karesi (4 seed × 15) → `results/test_predictions_locked.csv`
5. Test değerlendirmesi (görülmemiş karelere ayrı rapor), zamansal analiz

**Seçim kuralı (önceden sabit):** 4 seed'in doğrulama MAE ortalaması en düşük olan; eşitlikte seed'ler arası standart sapma, ortalama RMSE, karmaşıklık, süre (karmaşıklık süreden önce: süre ölçümü çalıştırmadan çalıştırmaya değişir).

> ⛔ Bölüm 4'ten sonra DURUN ve commit atın. Bölüm 5 test referanslarını ilk kez okur.

In [ ]:
# --- Kurulum: Colab'da repoyu klonlar; yerelde notebooks/ klasöründen çalıştırılabilir ---
import os, sys, subprocess
REPO_URL = "https://github.com/James007Bond-code/goruntu-isleme-hucre-sayimi-grup-odevi.git"
REPO_NAME = "goruntu-isleme-hucre-sayimi-grup-odevi"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.isdir("src"):
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
sys.path.insert(0, os.getcwd())
print("Çalışma klasörü:", os.getcwd())

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import download_dataset, load_image, read_gt_count
from src.selection import build_selection, load_selection
from src.pipeline import BASELINE, E11_SULEYMAN, make, param_str
from src.evaluation import (evaluate, warmup, run_experiments, rank_candidates,
                            evaluate_method, save_json)

os.makedirs(C.FIG_DIR, exist_ok=True); os.makedirs(C.FIG_IMG_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
n01, n02 = download_dataset()
print(f"Sekans 01: {n01} kare | Sekans 02: {n02} kare")

from src.selection import dev_pool_excluding_all_val, leaky_val_frames
from src.evaluation import lock_predictions, method_complexity, count_metrics

SEEDS = list(C.SEEDS.values())
sels = {s: load_selection(s) for s in SEEDS}
refs = {}
for s, df in sels.items():
    dv = df[df.group.isin(["Development", "Validation"])].copy()
    dv["gt_count"] = [read_gt_count(r.sequence, r.frame_idx) for r in dv.itertuples()]
    refs[s] = dv
methods = {k: json.load(open(os.path.join(C.seed_dir(s), f"method_{k}.json"))) for k, s in C.SEEDS.items()}
for k, m in methods.items():
    print(f"{k:9s} seed={m['seed']} -> {m['experiment']}")
warmup(load_image("01", sels[C.BASE_SEED].iloc[0].filename))

## Çapraz seed doğrulama matrisi
Satır: yöntemin sahibi, sütun: doğrulama seed'i. Yöntem sahibinin geliştirme karesi olan doğrulama kareleri o hücrede dışarıda bırakılır (`n_kare`).

In [ ]:
def val_frames(seed, exclude=()):
    v = refs[seed][refs[seed].group == "Validation"]
    return v[~v.frame_idx.isin(exclude)].reset_index(drop=True)

def cross_seed_table(method, owner_seed=None):
    rows = []
    for s in SEEDS:
        excl = leaky_val_frames(sels, owner_seed, s) if owner_seed is not None else []
        m = evaluate_method(val_frames(s, excl), method)
        rows.append({"val_seed": s, "MAE": m["MAE"], "RMSE": m["RMSE"], "Bias": m["Bias"],
                     "Sure_ms": m["Sure_ms"], "n_kare": 5 - len(excl), "haric": " ".join(map(str, excl))})
    return rows

BASELINE_M = {"name": "baseline", "config": BASELINE, "hm_ref": None}
cross = []
for k, m in [("baseline", BASELINE_M)] + list(methods.items()):
    for r in cross_seed_table(m, None if k == "baseline" else m["seed"]):
        cross.append({"yontem": k, **r})
df_cross = pd.DataFrame(cross)
df_cross.to_csv(os.path.join(C.RESULTS_DIR, "cross_seed_validation.csv"), index=False)
mat = df_cross.pivot(index="yontem", columns="val_seed", values="MAE")
mat["ortalama"] = mat.mean(axis=1); mat["std"] = mat[SEEDS].std(axis=1, ddof=0)
print("Doğrulama MAE – çapraz seed matrisi"); print(mat.round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 3.6))
im = ax.imshow(mat[SEEDS].to_numpy(), cmap="viridis_r")
ax.set_xticks(range(len(SEEDS)), SEEDS); ax.set_yticks(range(len(mat)), mat.index)
for i in range(len(mat)):
    for j in range(len(SEEDS)):
        ax.text(j, i, f"{mat.iloc[i, j]:.1f}", ha="center", va="center", color="w", fontsize=8)
ax.set_xlabel("Doğrulama seed'i"); ax.set_title("Çapraz seed doğrulama MAE", fontsize=10)
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/cross_seed_matrix.png", dpi=200); plt.show()

## 5. yöntem: dayanıklı pipeline
- **Ortak geliştirme havuzu:** 4 seed'in geliştirme kareleri birleşir; herhangi bir seed'in doğrulama karesiyle birebir aynı olanlar çıkarılır (`results/dev_pool_method5.csv`).
- Birleşik adaylar bu havuzda küçük bir taramayla ayarlanır; sonra 4 üye yöntemi + medyan topluluğu ile birlikte 4 seed doğrulamasında karşılaştırılır.

In [ ]:
pool, removed = dev_pool_excluding_all_val(sels)
pool_df = pd.DataFrame({"sequence": "01", "frame_idx": pool, "filename": [f"t{i:03d}.tif" for i in pool]})
pool_df["gt_count"] = [read_gt_count("01", i) for i in pool]
print(f"Ortak geliştirme havuzu: {len(pool)} kare | çıkarılan: {removed}")

# Birleşik aday ailesi (grup kararıyla genişletilebilir): CLAHE + Gauss + adaptif eşik + watershed ayırma
COMBO_BASE = make(E11_SULEYMAN, separation="watershed", ws_min_dist=16)
GRID = [make(COMBO_BASE, adapt_c=c, ws_min_dist=d) for c in (-8, -12, -16) for d in (12, 16, 20)]
tune = []
for cfg in GRID:
    m = evaluate(pool_df, cfg)
    tune.append({"Parametreler": param_str(cfg), "Pool_MAE": round(m["MAE"], 3), "Pool_RMSE": round(m["RMSE"], 3), "cfg": cfg})
df_tune = pd.DataFrame(tune).sort_values(["Pool_MAE", "Pool_RMSE"]).reset_index(drop=True)
df_tune.drop(columns="cfg").to_csv(os.path.join(C.RESULTS_DIR, "method5_tuning_pool.csv"), index=False)
print(df_tune.drop(columns="cfg").head(5).to_string(index=False))

M5_CANDIDATES = {
    **{f"M_{k}": m for k, m in methods.items()},
    "M5_combo": {"name": "M5_combo", "config": df_tune.loc[0, "cfg"], "hm_ref": None},
    "M5_ensemble_median": {"name": "M5_ensemble_median", "ensemble": list(methods.values()), "agg": "median"},
}
rows = []
for name, meth in M5_CANDIDATES.items():
    owner = meth.get("seed")
    per = cross_seed_table(meth, owner)
    if "ensemble" in meth:   # topluluk: her seed'de, o seed'in doğrulama karesini geliştirmede kullanan üyelerin kareleri hariç
        per = []
        for s in SEEDS:
            excl = sorted(set().union(*[leaky_val_frames(sels, m["seed"], s) for m in meth["ensemble"]]))
            mm = evaluate_method(val_frames(s, excl), meth)
            per.append({"val_seed": s, "MAE": mm["MAE"], "RMSE": mm["RMSE"], "Sure_ms": mm["Sure_ms"], "n_kare": 5 - len(excl)})
    p = pd.DataFrame(per)
    rows.append({"Aday": name, "Ort_Val_MAE": round(p.MAE.mean(), 3), "Std_Val_MAE": round(p.MAE.std(ddof=0), 3),
                 "Ort_Val_RMSE": round(p.RMSE.mean(), 3), "Ort_Sure_ms": round(p.Sure_ms.mean(), 1),
                 "Karmasiklik": method_complexity(meth), **{f"MAE_{s}": round(v, 2) for s, v in zip(p.val_seed, p.MAE)}})
df_m5 = pd.DataFrame(rows).sort_values(["Ort_Val_MAE", "Std_Val_MAE", "Ort_Val_RMSE", "Karmasiklik", "Ort_Sure_ms"]).reset_index(drop=True)
df_m5["Secim"] = ["SEÇİLDİ (5. yöntem)"] + ["elendi"] * (len(df_m5) - 1)
df_m5.to_csv(os.path.join(C.RESULTS_DIR, "method5_selection.csv"), index=False)
print(df_m5.to_string(index=False))
M5_NAME = df_m5.loc[0, "Aday"]

## 🔒 Kilit: nihai yöntem ve test tahminleri
Bu hücre test referanslarını **okumaz**.

> ⛔ Çalıştırdıktan sonra DURUN: `results/final_pipeline.json` + `results/test_predictions_locked.csv` + notebook → **ayrı commit**.

In [ ]:
LOCK_FILE = os.path.join(C.RESULTS_DIR, "test_predictions_locked.csv")
FINAL = {"baseline": BASELINE_M, **{f"M_{k}": m for k, m in methods.items()}, "M5": M5_CANDIDATES[M5_NAME]}
save_json({"selection_rule": "4 seed doğrulama MAE ortalaması; eşitlikte std, RMSE, karmaşıklık, süre",
           "method5_name": M5_NAME, "methods": FINAL, "seeds": C.SEEDS,
           "seen_test_seeds": sorted(C.SEEN_TEST_SEEDS)},
          os.path.join(C.RESULTS_DIR, "final_pipeline.json"))
test_df = pd.concat([sels[s][sels[s].group == "Test"] for s in SEEDS]).reset_index(drop=True)
locked = lock_predictions(test_df, FINAL, LOCK_FILE)
print(locked[[c for c in locked.columns if c.endswith("_pred")] + ["seed", "frame_idx"]].head(10).to_string(index=False))

## Bağımsız test (referanslar ilk kez burada okunur)
Seed 14048'in test kareleri bireysel çalışmada görülmüştü; özet tabloda **tüm kareler** ve **görülmemiş kareler** ayrı verilir.

In [ ]:
assert os.path.exists(LOCK_FILE)
res = pd.read_csv(LOCK_FILE, dtype={"sequence": str})
res.insert(5, "gt_count", [read_gt_count("02", i, lock_file=LOCK_FILE) for i in res.frame_idx])
seen = set()
for s in C.SEEN_TEST_SEEDS:
    seen |= set(sels[s].loc[sels[s].group == "Test", "frame_idx"])
res["gorulmus_kare"] = res.frame_idx.isin(seen)
names = [c[:-5] for c in res.columns if c.endswith("_pred")]
for n in names:
    res[f"{n}_abs_err"] = (res[f"{n}_pred"] - res["gt_count"]).abs()
res.to_csv(os.path.join(C.RESULTS_DIR, "test_results.csv"), index=False)

summ = []
for scope, sub in [("tum_kareler", res), ("gorulmemis_kareler", res[~res.gorulmus_kare])]:
    for s in SEEDS + ["hepsi"]:
        d = sub if s == "hepsi" else sub[sub.seed == s]
        if d.empty:
            continue
        for n in names:
            m = count_metrics(d[f"{n}_pred"], d["gt_count"])
            summ.append({"kapsam": scope, "seed": s, "yontem": n, "n_kare": len(d), **{k: round(v, 3) for k, v in m.items()},
                         "Sure_ms": round(d[f"{n}_time_ms"].mean(), 1)})
df_summ = pd.DataFrame(summ)
df_summ.to_csv(os.path.join(C.RESULTS_DIR, "test_summary.csv"), index=False)
print(df_summ[df_summ.seed == "hepsi"].to_string(index=False))
print(df_summ[(df_summ.kapsam == "tum_kareler") & (df_summ.yontem.isin(["baseline", "M5"]))]
      .pivot(index="yontem", columns="seed", values="MAE").to_string())

## Zamansal analiz (her seed için 3 blok)

In [ ]:
for s in SEEDS:
    d = res[res.seed == s]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
    for ax, (b, g) in zip(axes, d.groupby("block")):
        ax.plot(g.frame_idx, g.gt_count, "k-o", label="Gerçek")
        ax.plot(g.frame_idx, g.baseline_pred, "--s", color="#E53935", label="Baseline")
        ax.plot(g.frame_idx, g.M5_pred, "-^", color="#1E88E5", label="5. yöntem")
        ax.set_title(f"Seed {s} – blok {b}", fontsize=9); ax.set_xlabel("Kare"); ax.grid(alpha=0.3)
    axes[0].set_ylabel("Hücre sayısı"); axes[0].legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/fig5_temporal_{s}.png", dpi=200); plt.show()

temporal = res[["seed", "block", "frame_idx", "gt_count", "baseline_pred", "M5_pred"]].copy()
for c in ["gt_count", "baseline_pred", "M5_pred"]:
    temporal[f"d_{c}"] = temporal.groupby(["seed", "block"])[c].diff()
temporal.to_csv(os.path.join(C.RESULTS_DIR, "temporal_analysis.csv"), index=False)